# Malaria Parasite Detection from Cell ImagesComparative evaluation of MobileNet V3, ResNet-50, and VGG-16 for malaria parasite detection.**Dataset:** NIH Malaria Cell Images Dataset (27,558 images)  **Classes:** Parasitized, Uninfected  **Reference:** Rajaraman et al. (2018), PeerJ---

## 1. Environment Setup

In [ ]:
# Run on Google Colab or Kaggle with GPU enabled# !pip install tensorflow matplotlib seaborn scikit-learnimport os, json, numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersfrom tensorflow.keras.preprocessing.image import ImageDataGeneratorfrom tensorflow.keras.applications import MobileNetV3Large, ResNet50, VGG16from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenet_ppfrom tensorflow.keras.applications.resnet50 import preprocess_input as resnet_ppfrom tensorflow.keras.applications.vgg16 import preprocess_input as vgg_ppfrom tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateaufrom sklearn.metrics import (    classification_report, confusion_matrix,    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)print(f'TensorFlow: {tf.__version__}')print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2. Dataset Configuration

In [ ]:
# CONFIGURE THESE PATHS# Kaggle: TRAIN_DIR = '/kaggle/input/iarunava/cell-images-for-detecting-malaria/...'# Colab:  mount drive first, then set pathTRAIN_DIR = 'data/train'TEST_DIR = 'data/test'# Dataset: https://www.kaggle.com/datasets/iarunava/cell-images-for-detecting-malaria# Structure: cell_images/Parasitized, cell_images/UninfectedCLASSES = ['Parasitized', 'Uninfected']NUM_CLASSES = 2IMG_SIZE = (224, 224)BATCH_SIZE = 32EPOCHS = 15MODELS_OUT = 'models/malaria'METRICS_OUT = 'metrics/malaria'os.makedirs(MODELS_OUT, exist_ok=True)os.makedirs(METRICS_OUT, exist_ok=True)

## 3. Data Loading & Augmentation

In [ ]:
train_datagen = ImageDataGenerator(    rescale=1./255,    rotation_range=20,    width_shift_range=0.2,    height_shift_range=0.2,    shear_range=0.15,    zoom_range=0.15,    horizontal_flip=True,    fill_mode='nearest',    validation_split=0.15)test_datagen = ImageDataGenerator(rescale=1./255)train_gen = train_datagen.flow_from_directory(    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,    class_mode='binary', subset='training', seed=42, shuffle=True)val_gen = train_datagen.flow_from_directory(    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,    class_mode='binary', subset='validation', seed=42, shuffle=False)test_gen = test_datagen.flow_from_directory(    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,    class_mode='binary', shuffle=False)print(f'Train: {train_gen.samples}, Val: {val_gen.samples}, Test: {test_gen.samples}')print(f'Classes: {train_gen.class_indices}')

## 4. Model Builder

In [ ]:
def build_model(base_fn, name):    base = base_fn(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))    base.trainable = False    inputs = keras.Input(shape=(*IMG_SIZE, 3))    x = base(inputs, training=False)    x = layers.GlobalAveragePooling2D()(x)    x = layers.BatchNormalization()(x)    x = layers.Dense(256, activation='relu')(x)    x = layers.Dropout(0.4)(x)    x = layers.Dense(128, activation='relu')(x)    x = layers.Dropout(0.3)(x)    outputs = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)    model = keras.Model(inputs, outputs, name=name)    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])    return model, basedef fine_tune(model, base, lr=1e-5):    base.trainable = True    fine_tune_at = int(len(base.layers) * 0.7)    for layer in base.layers[:fine_tune_at]:        layer.trainable = False    model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])    return modelcallbacks = [    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)]

## 5. Train MobileNet V3

In [ ]:
print('=' * 60)print('TRAINING: MobileNet V3')print('=' * 60)mobilenet, mobilenet_base = build_model(MobileNetV3Large, 'mobilenet_v3')mobilenet.summary()# Phase 1: train headprint('\nPhase 1: Training classification head...')h1 = mobilenet.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)# Phase 2: fine-tuneprint('\nPhase 2: Fine-tuning...')mobilenet = fine_tune(mobilenet, mobilenet_base)h2 = mobilenet.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)mobilenet.save(os.path.join(MODELS_OUT, 'mobilenet_v3.keras'))print('Saved: mobilenet_v3.keras')

## 6. Train ResNet-50

In [ ]:
print('=' * 60)print('TRAINING: ResNet-50')print('=' * 60)resnet, resnet_base = build_model(ResNet50, 'resnet50')print('\nPhase 1: Training classification head...')h1 = resnet.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)print('\nPhase 2: Fine-tuning...')resnet = fine_tune(resnet, resnet_base)h2 = resnet.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)resnet.save(os.path.join(MODELS_OUT, 'resnet50.keras'))print('Saved: resnet50.keras')

## 7. Train VGG-16

In [ ]:
print('=' * 60)print('TRAINING: VGG-16')print('=' * 60)vgg, vgg_base = build_model(VGG16, 'vgg16')print('\nPhase 1: Training classification head...')h1 = vgg.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)print('\nPhase 2: Fine-tuning...')vgg = fine_tune(vgg, vgg_base)h2 = vgg.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)vgg.save(os.path.join(MODELS_OUT, 'vgg16.keras'))print('Saved: vgg16.keras')

## 8. Evaluation on Test Set

In [ ]:
def evaluate_model(model, model_name, model_key, generator):    generator.reset()    y_pred_probs = model.predict(generator, verbose=1)    if NUM_CLASSES == 2:        y_pred = (y_pred_probs > 0.5).astype(int).flatten()    else:        y_pred = np.argmax(y_pred_probs, axis=1)    y_true = generator.classes    class_names = list(generator.class_indices.keys())    acc = accuracy_score(y_true, y_pred)    prec = precision_score(y_true, y_pred, average='binary', zero_division=0)    rec = recall_score(y_true, y_pred, average='binary', zero_division=0)    f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)    try:        if NUM_CLASSES == 2:            auc = roc_auc_score(y_true, y_pred_probs.flatten())        else:            from sklearn.preprocessing import label_binarize            y_bin = label_binarize(y_true, classes=range(NUM_CLASSES))            auc = roc_auc_score(y_bin, y_pred_probs, multi_class='ovr', average='weighted')    except:        auc = 0.0    cm = confusion_matrix(y_true, y_pred).tolist()    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)    per_class = {}    for cn in class_names:        if cn in report:            per_class[cn] = {'precision': round(report[cn]['precision'], 3), 'recall': round(report[cn]['recall'], 3), 'f1': round(report[cn]['f1-score'], 3)}    print(f'\n{model_name}:')    print(f'  Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}')    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))    return {        'model_name': model_name, 'disease': 'malaria', 'dataset': 'NIH Malaria Cell Images Dataset',        'parameters': f'{model.count_params():,}',        'metrics': {'accuracy': round(acc,4), 'precision': round(prec,4), 'recall': round(rec,4), 'f1_score': round(f1,4), 'auc_roc': round(auc,4)},        'confusion_matrix': cm, 'class_labels': class_names, 'per_class_metrics': per_class    }, cm

In [ ]:
# Evaluate all modelsall_metrics = {}all_cms = {}for model, name, key in [(mobilenet, 'MobileNet V3', 'mobilenet_v3'), (resnet, 'ResNet-50', 'resnet50'), (vgg, 'VGG-16', 'vgg16')]:    m, cm = evaluate_model(model, name, key, test_gen)    all_metrics[key] = m    all_cms[key] = cm

## 9. Export Metrics JSON

In [ ]:
for mk, md_dict in all_metrics.items():    path = os.path.join(METRICS_OUT, f'{mk}_metrics.json')    with open(path, 'w') as f:        json.dump(md_dict, f, indent=2)    print(f'Saved: {path}')print('\nAll metrics exported.')

## 10. Comparison Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))fig.suptitle('Malaria Parasite Detection from Cell Images - Model Comparison', fontweight='bold')# Bar chartmetric_keys = ['accuracy','precision','recall','f1_score','auc_roc']metric_labels = ['Accuracy','Precision','Recall','F1','AUC-ROC']x = np.arange(len(metric_labels))w = 0.25colors = ['#4361ee','#7209b7','#f72585']for i, (mk, md_dict) in enumerate(all_metrics.items()):    vals = [md_dict['metrics'][k] for k in metric_keys]    axes[0].bar(x + i*w, vals, w, label=md_dict['model_name'], color=colors[i])axes[0].set_ylabel('Score')axes[0].set_title('Performance Metrics')axes[0].set_xticks(x + w)axes[0].set_xticklabels(metric_labels, rotation=45)axes[0].legend()axes[0].set_ylim(0, 1.05)# Confusion matrices for first 2 modelscls_names = list(test_gen.class_indices.keys())for idx, (mk, cm) in enumerate(list(all_cms.items())[:2]):    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cls_names, yticklabels=cls_names, ax=axes[idx+1])    axes[idx+1].set_title(f'{all_metrics[mk]["model_name"]}')    axes[idx+1].set_ylabel('Actual')    axes[idx+1].set_xlabel('Predicted')plt.tight_layout()plt.savefig('malaria_comparison.png', dpi=150, bbox_inches='tight')plt.show()

## 11. All Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))fig.suptitle('Confusion Matrices - All Models', fontweight='bold')cls_names = list(test_gen.class_indices.keys())for idx, (mk, cm) in enumerate(all_cms.items()):    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cls_names, yticklabels=cls_names, ax=axes[idx])    axes[idx].set_title(all_metrics[mk]['model_name'])    axes[idx].set_ylabel('Actual')    axes[idx].set_xlabel('Predicted')plt.tight_layout()plt.savefig('malaria_confusion_matrices.png', dpi=150, bbox_inches='tight')plt.show()

## 12. Summary

In [ ]:
print('=' * 70)print('SUMMARY')print('=' * 70)print(f'{"Model":<20s} {"Acc":>8s} {"Prec":>8s} {"Rec":>8s} {"F1":>8s} {"AUC":>8s}')print('-' * 52)for mk, md_dict in all_metrics.items():    m = md_dict['metrics']    print(f'{md_dict["model_name"]:<20s} {m["accuracy"]:>8.4f} {m["precision"]:>8.4f} {m["recall"]:>8.4f} {m["f1_score"]:>8.4f} {m["auc_roc"]:>8.4f}')print(f'\nModels saved to: {MODELS_OUT}/')print(f'Metrics saved to: {METRICS_OUT}/')print('\nCopy .keras and _metrics.json files to the Disease Prediction System project.')